In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mc
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter, NullFormatter, FuncFormatter

# Output directory for PDF exports
IMG_DIR = "."

# Global font sizes for print readability
plt.rcParams.update({
    'font.size':        14,
    'axes.titlesize':   16,
    'axes.labelsize':   14,
    'xtick.labelsize':  12,
    'ytick.labelsize':  12,
    'legend.fontsize':  12,
    'figure.titlesize': 16,
})


def darken(color, factor=0.65):
    """Return a darkened hex color by scaling each RGB component."""
    return mc.to_hex(tuple(c * factor for c in mc.to_rgb(color)))


def style_boxplot(bp, fill_color):
    """
    Style a single-color boxplot: fill at 50% alpha,
    whiskers/caps/medians/border in a darkened shade of fill_color.
    """
    border = darken(fill_color)
    for element in ["whiskers", "caps", "medians"]:
        for item in bp[element]:
            item.set_color(border)
            item.set_linewidth(1)
    for patch in bp["boxes"]:
        patch.set_facecolor((*mc.to_rgb(fill_color), 0.5))
        patch.set_edgecolor(border)
        patch.set_linewidth(1)

# 1 · Throughput: Locking Strategies

In [ ]:
# Static configuration
WARMUP_SECONDS = 5
LOCK_COLORS    = {
    "FOR UPDATE": "#FF7C58",
    "FOR UPDATE SKIP LOCKED": "#2D9A7A",
}

# Load CSV
lock_results = pd.read_csv("csv/throughput_locking_results.csv",
                            parse_dates=["created_at", "completed_at"])
lock_metrics = pd.read_csv("csv/throughput_locking_metrics.csv",
                            parse_dates=["timestamp"])

# Compute throughput, skipping the warmup window
rows = []
for (mode, worker_count), group in lock_results.groupby(["lock_mode", "worker_count"]):
    warmup_cutoff = group["created_at"].min() + pd.Timedelta(seconds=WARMUP_SECONDS)
    measured      = group[group["completed_at"] > warmup_cutoff]
    if measured.empty:
        continue
    duration   = (measured["completed_at"].max() - warmup_cutoff).total_seconds()
    throughput = len(measured) / duration if duration > 0 else 0
    rows.append({"lock_mode": mode, "workers": worker_count,
                 "throughput": throughput, "jobs": len(measured)})

lock_df      = pd.DataFrame(rows)
worker_ticks = sorted(lock_df["workers"].unique())
worker_x_pos = {w: i for i, w in enumerate(worker_ticks)}  # worker count --> plot * index

# Compute worker-state metric
lock_metrics["elapsed_s"] = lock_metrics.groupby("case_id")["timestamp"].transform(
    lambda x: (x - x.min()).dt.total_seconds()
)
lock_states = lock_metrics.groupby(["lock_mode", "worker_count"], as_index=False).agg(
    workers_idle=("workers_idle", "mean"),
    workers_fetching=("workers_fetching", "mean"),
    workers_processing=("workers_processing", "mean"),
)
lock_states["pct_fetching"]   = lock_states["workers_fetching"]   / lock_states["worker_count"] * 100
lock_states["pct_processing"] = lock_states["workers_processing"] / lock_states["worker_count"] * 100
lock_states = lock_states.merge(
    lock_df[["lock_mode", "workers", "throughput"]],
    left_on=["lock_mode", "worker_count"],
    right_on=["lock_mode", "workers"],
)

for mode in lock_df["lock_mode"].unique():
    print(f"\n--- {mode} ---")
    display(lock_df[lock_df["lock_mode"] == mode][["workers", "throughput", "jobs"]].reset_index(drop=True))

In [ ]:
# Throughput vs. worker count
fig, ax = plt.subplots(figsize=(10, 3))

for mode, color in LOCK_COLORS.items():
    d = lock_df[lock_df["lock_mode"] == mode].sort_values("workers")
    ax.plot([worker_x_pos[w] for w in d["workers"]], d["throughput"],
            marker="o", linewidth=2.5, markersize=8, label=mode, color=color)

ax.set_xlabel("Worker Count")
ax.set_ylabel("Throughput (Jobs/s)")
ax.set_title("Throughput vs. Worker Count")
ax.set_xticks(range(len(worker_ticks)))
ax.set_xticklabels([str(int(w)) for w in worker_ticks])
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/throughput_vs_worker_count.pdf", bbox_inches="tight")
print(f"Saved: {IMG_DIR}/throughput_vs_worker_count.pdf")
plt.show()

In [ ]:
# Speedup of SKIP LOCKED over FOR UPDATE at each worker count
pivot = lock_df.pivot(index="workers", columns="lock_mode", values="throughput")
pivot["speedup"] = pivot["FOR UPDATE SKIP LOCKED"] / pivot["FOR UPDATE"]

fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(range(len(pivot)), pivot["speedup"],
       color="#2D9A7A", alpha=0.5, edgecolor=darken("#2D9A7A"), linewidth=1.5)

for i, (_, row) in enumerate(pivot.iterrows()):
    ax.text(i, row["speedup"] + 0.1, f"{row['speedup']:.1f}x",
            ha="center", va="bottom", fontweight="bold")

ax.axhline(y=1, color="#FF7C58", linestyle="-", linewidth=1.5, label="Equal speed")
ax.set_xticks(range(len(pivot)))
ax.set_xticklabels([str(int(w)) for w in pivot.index])
ax.set_xlabel("Worker Count")
ax.set_ylabel("Speedup Factor")
ax.set_title("SKIP LOCKED vs. FOR UPDATE")
ax.set_ylim(0, pivot["speedup"].max() * 1.15)
ax.legend(loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/speedup_skip_locked_vs_for_update.pdf", bbox_inches="tight")
print(f"Saved: {IMG_DIR}/speedup_skip_locked_vs_for_update.pdf")
plt.show()

display(pivot.round(2))

In [ ]:
# Worker-state summary
display(
    lock_states[["lock_mode", "worker_count", "workers_fetching", "workers_processing",
                 "pct_fetching", "pct_processing", "throughput"]].round(1)
)

def plot_state_metric(column, ylabel, title, filename):
    """Line plot of a lock_states metric over worker count for each locking mode."""
    fig, ax = plt.subplots(figsize=(10, 3))
    for mode, color in LOCK_COLORS.items():
        d = lock_states[lock_states["lock_mode"] == mode].sort_values("worker_count")
        ax.plot([worker_x_pos[w] for w in d["worker_count"]], d[column],
                marker="o", linewidth=2.5, markersize=8, label=mode, color=color)
    ax.set_xlabel("Worker Count")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(range(len(worker_ticks)))
    ax.set_xticklabels([str(int(w)) for w in worker_ticks])
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(f"{IMG_DIR}/{filename}", bbox_inches="tight")
    print(f"Saved: {IMG_DIR}/{filename}")
    plt.show()


plot_state_metric("workers_fetching", "Avg Workers in Dequeue", "Lock Contention",
                  "lock_contention.pdf")

# 2 · Latency: Polling vs. LISTEN/NOTIFY

In [ ]:
# Static configuration
LAT_STYLES = {
    "LISTEN/NOTIFY":       {"color": "#B082FF", "marker": "s"},
    "Lottery (1 Winner)":  {"color": "#3A4580", "marker": "^"},
    "Lottery (5 Winners)": {"color": "#2D9A7A", "marker": "^"},
    "Polling (50ms)":      {"color": "#FFB30B", "marker": "o"},
    "Polling (1000ms)":    {"color": "#FF7C58", "marker": "o"},
}

# Order in which modes appear in each box-plot
POLLING_VS_LISTEN_ORDER = ["LISTEN/NOTIFY", "Polling (50ms)", "Polling (1000ms)"]
LISTEN_VARIANTS_ORDER   = ["LISTEN/NOTIFY", "Lottery (5 Winners)", "Lottery (1 Winner)"]
IDLE_LOAD_BAR_ORDER     = ["LISTEN/NOTIFY", "Lottery (5 Winners)", "Lottery (1 Winner)",
                            "Polling (50ms)", "Polling (1000ms)"]

# Map internal CSV labels to display labels
LABEL_MAP = {
    "listening":              "LISTEN/NOTIFY",
    "listening_lottery_1win": "Lottery (1 Winner)",
    "listening_lottery_5win": "Lottery (5 Winners)",
}

def display_label(raw):
    """Convert raw CSV label to a readable display name."""
    if raw in LABEL_MAP:
        return LABEL_MAP[raw]
    if raw.startswith("polling_"):
        return f"Polling ({raw.replace('polling_', '')})"
    return raw


# Load CSV and prepare data
lat_df = pd.read_csv(
    "csv/latency_polling_vs_listen_results.csv",
    parse_dates=["created_at", "enqueued_at", "dequeued_at", "completed_at"],
)
numeric_cols = ["latency_ms", "worker_count", "polling_interval_ms",
                "lottery", "notify_winners", "dequeue_calls", "dequeue_empty"]
for col in numeric_cols:
    if col in lat_df.columns:
        lat_df[col] = pd.to_numeric(lat_df[col], errors="coerce")

lat_df["display_label"]  = lat_df["label"].apply(display_label)
lat_worker_counts        = sorted(lat_df["worker_count"].dropna().astype(int).unique())

# Compute idle-load stats
case_cols      = ["worker_mode", "label", "display_label", "worker_count",
                  "dequeue_calls", "dequeue_empty"]
lat_case_stats = lat_df[case_cols].drop_duplicates()
lat_case_stats["empty_rate"] = (
    lat_case_stats["dequeue_empty"] / lat_case_stats["dequeue_calls"]
).fillna(0)

lat_idle = lat_case_stats.groupby(["worker_count", "display_label"]).agg(
    dequeue_calls=("dequeue_calls", "first"),
    dequeue_empty=("dequeue_empty", "first"),
    empty_rate=("empty_rate", "first"),
).reset_index()
lat_idle["empty_rate_pct"] = (lat_idle["empty_rate"] * 100).round(1)
lat_idle.columns = ["Workers", "Mode", "Dequeue Calls", "Dequeue Empty", "Empty Rate", "Empty Rate (%)"]

# Latency statistics
print(f"Records: {len(lat_df)} | Worker counts: {lat_worker_counts}")

lat_stats = lat_df.groupby(["worker_count", "display_label"])["latency_ms"].agg([
    ("Count", "count"), ("Min", "min"), ("Max", "max"), ("Mean", "mean"),
    ("Median", "median"), ("Std", "std"),
    ("P95", lambda x: x.quantile(0.95)),
    ("P99", lambda x: x.quantile(0.99)),
]).round(2).reset_index()
lat_stats.columns = ["Workers", "Mode", "Count", "Min", "Max", "Mean", "Median", "Std", "P95", "P99"]

for wc in lat_worker_counts:
    print(f"\nWorkers {wc}")
    display(lat_stats[lat_stats["Workers"] == wc][
        ["Mode", "Count", "Min", "Max", "Mean", "Median", "Std", "P95", "P99"]
    ])

In [ ]:
# Latency box plots – all worker counts in one figure per config, shared legend
import matplotlib.patches as mpatches

box_plot_configs = [
    ("Polling vs. LISTEN/NOTIFY",  POLLING_VS_LISTEN_ORDER, [1, 10, 100, 1000]),
    ("LISTEN/NOTIFY Variants",      LISTEN_VARIANTS_ORDER,   [1, 10, 100, 1000]),
]

n = len(lat_worker_counts)
subplot_size = 4  # square subplots

for title_prefix, mode_order, ytick_values in box_plot_configs:
    fig, axes = plt.subplots(1, n, figsize=(subplot_size * n, subplot_size), sharey=True)
    legend_handles = []

    for ax, wc in zip(axes, lat_worker_counts):
        modes_data = [
            (m, lat_df[(lat_df["display_label"] == m) &
                       (lat_df["worker_count"] == wc)]["latency_ms"].dropna().values)
            for m in mode_order
            if m in lat_df["display_label"].unique()
        ]
        modes_data = [(m, d) for m, d in modes_data if len(d) > 0]
        if not modes_data:
            continue

        modes, data = zip(*modes_data)
        bp = ax.boxplot(list(data), labels=[""] * len(modes), patch_artist=True,
                        showfliers=False, widths=0.5)

        legend_handles.clear()
        for i, (patch, mode) in enumerate(zip(bp["boxes"], modes)):
            color  = LAT_STYLES[mode]["color"]
            border = darken(color)
            patch.set_facecolor((*mc.to_rgb(color), 0.5))
            patch.set_edgecolor(border)
            patch.set_linewidth(1)
            for item in [bp["whiskers"][2*i], bp["whiskers"][2*i+1],
                         bp["caps"][2*i],     bp["caps"][2*i+1]]:
                item.set_color(border)
                item.set_linewidth(1)
            bp["medians"][i].set_color(border)
            bp["medians"][i].set_linewidth(1)
            legend_handles.append(
                mpatches.Patch(facecolor=(*mc.to_rgb(color), 0.5), edgecolor=border, label=mode)
            )

        ax.set_title(f"{wc} Worker(s)")
        ax.set_yscale("log")
        ax.set_yticks(ytick_values)
        ax.set_yticklabels([str(v) for v in ytick_values])
        ax.grid(True, axis="y", alpha=0.3, which="both")
        ax.tick_params(axis="x", which="both", bottom=False)

    axes[0].set_ylabel("Latency (ms) [log]")

    fig.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, 0.05),
               ncol=len(legend_handles), frameon=True, fontsize=11)
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.2)

    slug = title_prefix.lower().replace(" ", "_").replace(".", "").replace("/", "_")
    fname = f"{IMG_DIR}/boxplot_latency_{slug}.pdf"
    fig.savefig(fname, bbox_inches="tight")
    print(f"Saved: {fname}")
    plt.show()

In [ ]:
# Idle-load summary table
for wc in lat_worker_counts:
    print(f"\nWorkers {wc}")
    display(lat_idle[lat_idle["Workers"] == wc][
        ["Mode", "Dequeue Calls", "Dequeue Empty", "Empty Rate (%)"]
    ])

In [ ]:
# Empty dequeues – all worker counts in one figure with shared legend
import matplotlib.patches as mpatches

jobs_per_wc = lat_df.groupby(["worker_count", "display_label"]).size().reset_index(name="n_jobs")

n = len(lat_worker_counts)
subplot_size = 4  # square subplots

fig, axes = plt.subplots(1, n, figsize=(subplot_size * n, subplot_size), sharey=False)
legend_handles = []

for ax, wc in zip(axes, lat_worker_counts):
    subset = lat_idle[lat_idle["Workers"] == wc].copy()
    if subset.empty:
        continue

    n_jobs_rows = jobs_per_wc[jobs_per_wc["worker_count"] == wc]["n_jobs"]
    n_jobs = n_jobs_rows.iloc[0] if not n_jobs_rows.empty else "?"

    subset["sort_key"] = subset["Mode"].apply(
        lambda m: IDLE_LOAD_BAR_ORDER.index(m) if m in IDLE_LOAD_BAR_ORDER else 99
    )
    subset = subset.sort_values("sort_key")
    x      = np.arange(len(subset))

    colors      = [LAT_STYLES.get(m, {}).get("color", "#999999") for m in subset["Mode"]]
    face_colors = [(*mc.to_rgb(c), 0.5) for c in colors]
    edge_colors = [darken(c) for c in colors]
    max_val     = subset["Dequeue Empty"].max()

    bars = ax.bar(x, subset["Dequeue Empty"], color=face_colors, edgecolor=edge_colors, linewidth=1)

    for bar, val in zip(bars, subset["Dequeue Empty"]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max_val * 0.02,
                f"{int(val)}", ha="center", va="bottom", fontsize=10)

    ax.set_title(f"{wc} Worker(s)")
    ax.set_ylim(0, max_val * 1.15)
    ax.set_xticks(x)
    ax.set_xticklabels([""] * len(subset))
    ax.tick_params(axis="x", which="both", bottom=False)
    ax.grid(True, axis="y", alpha=0.3)

    if not legend_handles:
        legend_handles = [
            mpatches.Patch(
                facecolor=(*mc.to_rgb(LAT_STYLES.get(m, {}).get("color", "#999999")), 0.5),
                edgecolor=darken(LAT_STYLES.get(m, {}).get("color", "#999999")),
                label=m
            )
            for m in subset["Mode"]
        ]

axes[0].set_ylabel("Empty Dequeues")

fig.legend(handles=legend_handles, loc="lower center", bbox_to_anchor=(0.5, -0.05),
           ncol=len(legend_handles), frameon=True, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(bottom=0.2)

fname = f"{IMG_DIR}/empty_dequeues.pdf"
fig.savefig(fname, bbox_inches="tight")
print(f"Saved: {fname}")
plt.show()

In [ ]:
# Latency vs. idle load – all worker counts in one figure with shared legend
all_latencies   = lat_df["latency_ms"].dropna().values
all_empty_rates = [
    lat_case_stats[(lat_case_stats["display_label"] == m) &
                   (lat_case_stats["worker_count"] == wc)]["empty_rate"].iloc[0] * 100
    for wc in lat_worker_counts
    for m in lat_df["display_label"].unique()
    if not lat_case_stats[(lat_case_stats["display_label"] == m) &
                           (lat_case_stats["worker_count"] == wc)].empty
]

y_min = max(0.5, all_latencies.min() * 0.5)
y_max = all_latencies.max() * 1.5
x_min = max(0, min(all_empty_rates) - 5)
x_max = max(all_empty_rates) + 5

n = len(lat_worker_counts)
subplot_size = 4  # square subplots
fig, axes = plt.subplots(1, n, figsize=(subplot_size * n, subplot_size), sharey=True)

for ax, wc in zip(axes, lat_worker_counts):
    handles, labels = [], []

    for mode in lat_df["display_label"].unique():
        latencies = lat_df[(lat_df["display_label"] == mode) &
                           (lat_df["worker_count"] == wc)]["latency_ms"]
        case_row  = lat_case_stats[(lat_case_stats["display_label"] == mode) &
                                    (lat_case_stats["worker_count"] == wc)]
        if latencies.empty or case_row.empty:
            continue

        style  = LAT_STYLES.get(mode, {"color": "#999999", "marker": "o"})
        color  = style["color"]
        sc = ax.scatter(
            case_row["empty_rate"].iloc[0] * 100,
            latencies.median(),
            color=(*mc.to_rgb(color), 0.5),
            marker=style["marker"],
            s=250,
            edgecolors=darken(color),
            linewidth=1,
        )
        handles.append(sc)
        labels.append(mode)

    ax.set_xlabel("Idle Load (Empty Rate %)")
    ax.set_title(f"{wc} Worker(s)")
    ax.set_yscale("log")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_yticks([1, 10, 100, 1000])
    ax.set_yticklabels(["1", "10", "100", "1000"])
    ax.grid(True, alpha=0.3, which="both")
    ax.set_aspect("auto")
    ax.text(0.02, 0.02, "Low Latency,\nLow Load",
            transform=ax.transAxes, fontsize=11, color="green", va="bottom")
    ax.text(0.98, 0.98, "High Latency,\nHigh Load",
            transform=ax.transAxes, fontsize=11, color="red", va="top", ha="right")

axes[0].set_ylabel("Median Latency (ms) [log]")

fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.53, -0.05),
           ncol=len(handles), frameon=True, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(bottom=0.25)

fname = f"{IMG_DIR}/latency_vs_idle_load.pdf"
fig.savefig(fname, bbox_inches="tight")
print(f"Saved: {fname}")
plt.show()

# 3 · Retry Strategies

In [ ]:
# Static configuration
STRATEGY_ORDER = ["constant", "linear", "exponential"]
JITTER_ORDER   = ["none", "full jitter", "equal jitter"]

JITTER_COLORS = {"none": "#FF7C58", "full jitter": "#2D9A7A", "equal jitter": "#B082FF"}
JITTER_LABELS = {"none": "No Jitter",    "full jitter": "Full Jitter",    "equal jitter": "Equal Jitter"}

# Theoretical delays (ms) per retry attempt for each strategy (based on the scenarios)
BASE_DELAYS = {
    "constant":    [100, 100, 100],
    "linear":      [200, 300, 400],
    "exponential": [200, 400, 800],
}

ATTEMPT_COLORS  = {1: "#FF7C58", 2: "#FFB30B", 3: "#2D9A7A", 4: "#B082FF", 5: "#3A4580"}
N_SCATTER_JOBS  = 150   # sampled jobs per panel in the event scatter

COLOR_THEORETICAL = "#3A4580"
COLOR_MEASURED    = "#FF7C58"

# Box-plot layout parameters
BOX_OFFSET    = 0.2  # horizontal offset between theoretical and measured box
GROUP_SPACING = 1.1  # x-distance between jitter groups within one strategy
STRATEGY_GAP  = 1.0   # extra x-gap inserted between strategies


# Helper functions
def theoretical_mean(delays, jitter):
    """Analytical expected mean delay for a given backoff + jitter combination."""
    if   jitter == "none":         return np.mean(delays)
    elif jitter == "full jitter":  return np.mean([d / 2    for d in delays])  # U[1,d] ≈ d/2
    elif jitter == "equal jitter": return np.mean([d * 0.75 for d in delays])  # U[d/2,d] → 3d/4


def sample_theoretical_delays(delays, jitter, n_per_attempt=1000):
    """
    Sample n theoretical delays per attempt to build a comparison distribution.
    Mirrors the backoff formulas used in production code.
    """
    samples = []
    for d in delays:
        if   jitter == "none":         samples.extend([float(d)] * n_per_attempt)
        elif jitter == "full jitter":  samples.extend(np.random.uniform(1, d, n_per_attempt))
        elif jitter == "equal jitter": samples.extend(np.random.uniform(d / 2, d, n_per_attempt))
    return np.array(samples)


# Load and prepare data
np.random.seed(42)  # reproducible sampling for all subsequent plots

retry_df = pd.read_csv("csv/retry_strategies_results.csv", parse_dates=["attempted_at"])

# attempt > 1: actual retries; retry_delay_ms > 0: exclude the final failure row (no next delay)
retries = retry_df[(retry_df["attempt"] > 1) & (retry_df["retry_delay_ms"] > 0)].copy()

# Pre-compute per-job relative timestamps for the scatter plot
retry_df["first_ms"] = retry_df.groupby(
    ["strategy_type", "jitter", "job_id"]
)["attempted_at_ms"].transform("min")
retry_df["rel_ms"] = retry_df["attempted_at_ms"] - retry_df["first_ms"]

print(f"Raw data: {len(retry_df)} | Retries (filtered): {len(retries)}\n")

stats_retry = retries.groupby(["strategy_type", "jitter"])["retry_delay_ms"].agg(
    n="count", min="min", mean="mean", median="median", max="max", std="std"
).round(1)
stats_retry = stats_retry.reindex(STRATEGY_ORDER, level=0).reindex(JITTER_ORDER, level=1)
display(stats_retry)

In [ ]:
# Box plot: theoretical vs. measured delay distributions, log scale

# Build x-positions and data arrays for all (strategy, jitter) combinations
theo_positions, meas_positions = [], []
theo_data,      meas_data      = [], []
tick_positions, tick_labels    = [], []
strategy_label_x = {}  # center x for each strategy label
divider_x        = []  # x positions for vertical separators between strategies

x = 0
for strategy in STRATEGY_ORDER:
    jitter_x_positions = []
    for jitter in JITTER_ORDER:
        theo_positions.append(x - BOX_OFFSET)
        meas_positions.append(x + BOX_OFFSET)
        theo_data.append(sample_theoretical_delays(BASE_DELAYS[strategy], jitter))
        meas_data.append(
            retries[(retries["strategy_type"] == strategy) &
                    (retries["jitter"] == jitter)]["retry_delay_ms"].values
        )
        tick_positions.append(x)
        tick_labels.append(JITTER_LABELS[jitter])
        jitter_x_positions.append(x)
        x += GROUP_SPACING
    strategy_label_x[strategy] = np.mean(jitter_x_positions)
    divider_x.append(x - GROUP_SPACING / 2 + STRATEGY_GAP / 2)
    x += STRATEGY_GAP

fig, ax = plt.subplots(figsize=(16, 6))

bp_theo = ax.boxplot(theo_data, positions=theo_positions, widths=0.35,
                     patch_artist=True, showfliers=False)
bp_meas = ax.boxplot(meas_data, positions=meas_positions, widths=0.35,
                     patch_artist=True, showfliers=False)

style_boxplot(bp_theo, COLOR_THEORETICAL)
style_boxplot(bp_meas, COLOR_MEASURED)

ax.set_yscale("log")
ax.set_ylabel("Retry Delay (ms) [log]", fontsize=20)
ax.set_yticks([1, 10, 25, 50, 100, 200, 400, 800])
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.yaxis.set_minor_formatter(NullFormatter())
ax.tick_params(axis="y", labelsize=18)
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, fontsize=18)
ax.grid(True, axis="y", alpha=0.3, which="both")

# Strategy labels below x-axis and vertical dividers between them
for strategy, center_x in strategy_label_x.items():
    ax.text(center_x, -0.12, strategy.capitalize(),
            ha="center", va="top", fontsize=20, fontweight="bold",
            transform=ax.get_xaxis_transform())
for sep_x in divider_x[:-1]:
    ax.axvline(sep_x, color="gray", linestyle="--", linewidth=1, alpha=0.75)

ax.legend(handles=[
    Patch(facecolor=(*mc.to_rgb(COLOR_THEORETICAL), 0.5),
          edgecolor=darken(COLOR_THEORETICAL), linewidth=1, label="Theoretical"),
    Patch(facecolor=(*mc.to_rgb(COLOR_MEASURED), 0.5),
          edgecolor=darken(COLOR_MEASURED), linewidth=1, label="Measured"),
], loc="upper left", fontsize=16)

ax.set_title("Theoretical vs. Measured Backoffs", fontsize=20)
plt.tight_layout()
fig.savefig(f"{IMG_DIR}/boxplot_retry_theoretical_vs_measured.pdf", bbox_inches="tight")
print(f"Saved: {IMG_DIR}/boxplot_retry_theoretical_vs_measured.pdf")
plt.show()

In [ ]:
# Retry timing per job, sampled to N_SCATTER_JOBS per panel
fig, axes = plt.subplots(3, 3, figsize=(18, 10), sharex=True, sharey=True)

for row, strategy in enumerate(STRATEGY_ORDER):
    for col, jitter in enumerate(JITTER_ORDER):
        ax  = axes[row, col]
        sub = retry_df[(retry_df["strategy_type"] == strategy) &
                       (retry_df["jitter"] == jitter)].copy()

        sampled_ids = np.random.choice(sub["job_id"].unique(), size=N_SCATTER_JOBS, replace=False)
        sub         = sub[sub["job_id"].isin(sampled_ids)].copy()
        sub["job_idx"] = sub["job_id"].map({jid: i for i, jid in enumerate(sampled_ids)})

        for attempt, color in ATTEMPT_COLORS.items():
            pts = sub[sub["attempt"] == attempt]
            if not pts.empty:
                ax.scatter(pts["rel_ms"], pts["job_idx"],
                           s=18, alpha=1, marker="o", color=color, linewidths=1)

        ax.grid(True, alpha=0.5)
        ax.set_yticks([])
        ax.tick_params(axis="x", labelsize=20)
        if row == 0: ax.set_title(JITTER_LABELS[jitter], fontsize=20)
        if col == 0: ax.set_ylabel(f"{strategy.capitalize()}\n\nJob", fontsize=20)
        if row == 2: ax.set_xlabel("Time since first attempt (ms)", fontsize=18, labelpad=16)

legend_handles = [
    Line2D([0], [0], marker="o", color=c, markersize=8, linewidth=0, label=f"Attempt {a}")
    for a, c in ATTEMPT_COLORS.items()
]
fig.legend(handles=legend_handles, loc="lower center", ncol=len(ATTEMPT_COLORS),
           fontsize=20, frameon=True, bbox_to_anchor=(0.5, -0.1))

plt.suptitle(f"All Retry Strategies, {N_SCATTER_JOBS} jobs per chart", fontsize=24)
plt.tight_layout()
plt.subplots_adjust(bottom=0.06)
fig.savefig(f"{IMG_DIR}/retry_timing_scatter.pdf", bbox_inches="tight")
print(f"Saved: {IMG_DIR}/retry_timing_scatter.pdf")
plt.show()

# 4 · DLQ Consistency: No Jobs Lost

In [ ]:
# Static configuration
# Four scenarios: outage (yes/no), error rate (0% vs 100%)
SCENARIO_LABELS = {
    "no_outage_error_rate_0":    "No Outage, 0% Errors",
    "no_outage_error_rate_100":  "No Outage, 100% Errors",
    "with_outage_error_rate_0":  "Outage, 0% Errors",
    "with_outage_error_rate_100":"Outage, 100% Errors",
}
SCENARIO_ORDER = list(SCENARIO_LABELS.keys())

COLOR_COMPLETED = "#2D9A7A"  # jobs processed successfully
COLOR_DLQ       = "#FF7C58"  # jobs moved to DLQ after max retries
ATTEMPT_PALETTE = {1: "#2D9A7A", 2: "#FFB30B", 5: "#FF7C58"}  # attempt counts seen in data

# ── Load data ────────────────────────────────────────────────────────────────
dlq_df = pd.read_csv(
    "csv/dlq_consistency_results.csv",
    parse_dates=["created_at", "enqueued_at", "dequeued_at", "completed_at", "failed_at"],
)

# Collapse to one summary row per scenario (case-level fields repeat per job row)
dlq_summary = (
    dlq_df.groupby("case_label")
    .agg(total=("total_jobs", "first"),
         completed=("completed_jobs", "first"),
         dlq=("failed_jobs", "first"),
         lost=("lost_jobs", "first"))
    .reindex(SCENARIO_ORDER)
    .reset_index()
)
dlq_summary["label"] = dlq_summary["case_label"].map(SCENARIO_LABELS)

# Attempt distribution per scenario (how many jobs needed 1, 2, or 5 attempts)
attempt_counts = (
    dlq_df.groupby(["case_label", "attempts"])
    .size()
    .unstack(fill_value=0)
    .reindex(SCENARIO_ORDER)
)

display(
    dlq_summary[["label", "total", "completed", "dlq", "lost"]]
    .rename(columns={"label": "Scenario", "total": "Total",
                     "completed": "Completed", "dlq": "DLQ", "lost": "Lost"})
)

In [ ]:
# Invariant checks – property-based verification for Research Question 5.
# Each function receives (scenario_df, summary_row) and returns (passed: bool, detail: str).
MAX_ATTEMPTS = 5  # configured retry cap in the benchmark


def check_accounting(df, row):
    """total_jobs == completed + dlq + lost  →  no ghost or vanished jobs."""
    delta = row["total"] - row["completed"] - row["dlq"] - row["lost"]
    return delta == 0, f"delta={int(delta)}"


def check_no_lost_jobs(df, row):
    """lost_jobs is zero – at-least-once delivery guarantee holds."""
    return int(row["lost"]) == 0, f"lost={int(row['lost'])}"


def check_job_uniqueness(df, row):
    """Each job_id appears exactly once – no job was processed twice."""
    dupes = df["job_id"].duplicated().sum()
    return dupes == 0, f"duplicates={dupes}"


def check_retry_cap(df, row):
    """No job exceeded MAX_ATTEMPTS – retry policy is enforced."""
    violations = (df["attempts"] > MAX_ATTEMPTS).sum()
    return violations == 0, f"violations={violations}"


def check_terminal_coverage(df, row):
    """Every job has a non-null status – nothing is stuck in an undefined state."""
    nulls = df["status"].isna().sum()
    return nulls == 0, f"nulls={nulls}"


def check_status_consistency(df, row):
    """Row-level status counts match the aggregated summary columns."""
    n_completed = (df["status"] == "completed").sum()
    n_dlq       = (df["status"] != "completed").sum()   # non-completed ≙ DLQ
    match = (n_completed == row["completed"]) and (n_dlq == row["dlq"])
    return match, (f"rows: completed={n_completed}/{int(row['completed'])}, "
                   f"dlq={n_dlq}/{int(row['dlq'])}")


CHECKS = [
    ("Accounting",         check_accounting),
    ("No Lost Jobs",       check_no_lost_jobs),
    ("Job Uniqueness",     check_job_uniqueness),
    ("Retry Cap (≤5)",     check_retry_cap),
    ("Terminal Coverage",  check_terminal_coverage),
    ("Status Consistency", check_status_consistency),
]

# Run every invariant against every scenario
result_rows = []
for _, summary_row in dlq_summary.iterrows():
    scenario_df = dlq_df[dlq_df["case_label"] == summary_row["case_label"]]
    for check_name, fn in CHECKS:
        passed, detail = fn(scenario_df, summary_row)
        result_rows.append({
            "Scenario": summary_row["label"],
            "Check":    check_name,
            "Result":   "PASS" if passed else "FAIL",
            "Detail":   detail,
        })

results_df   = pd.DataFrame(result_rows)
scenario_order = dlq_summary["label"].tolist()
pivot          = results_df.pivot(index="Check", columns="Scenario", values="Result")
pivot          = pivot[scenario_order]   # enforce display order

all_pass = (results_df["Result"] == "PASS").all()
print(f"Overall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}\n")
display(pivot)
print("\nDetail log:")
display(results_df[["Scenario", "Check", "Result", "Detail"]])